<a href="https://colab.research.google.com/github/nivedhithar2802-png/Food-Waste-Prediction-/blob/main/foodwastedashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import files

uploaded = files.upload()


Saving food_waste_model.pkl to food_waste_model.pkl


In [ ]:

%pip install -q gradio


In [ ]:

import joblib
import pandas as pd
import sklearn

print("Scikit-learn version:", sklearn.__version__)

model = joblib.load("food_waste_model.pkl")

print("Food waste prediction model loaded successfully!")


Scikit-learn version: 1.6.1
Food waste prediction model loaded successfully!


In [ ]:

import gradio as gr
import pandas as pd
import plotly.graph_objects as go

def predict_waste(day, menu_type, attendance, weather, meals_prepared):
    # Prepare input in the exact format used for training
    input_data = pd.DataFrame([{
        "day": day,
        "menu_type": menu_type,
        "attendance": int(attendance),
        "weather": weather,
        "meals_prepared": int(meals_prepared)
    }])

    # Predict risk
    prediction = str(model.predict(input_data)[0])

    # Get class probabilities, if available
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(input_data)[0]
        classes = model.classes_

        chart = go.Figure(
            go.Bar(
                x=[str(c) for c in classes],
                y=[float(p) * 100 for p in probabilities],
                text=[f"{float(p)*100:.1f}%" for p in probabilities],
                textposition="auto"
            )
        )
        chart.update_layout(
            title="Model Prediction Probabilities",
            xaxis_title="Risk Category",
            yaxis_title="Probability (%)",
            yaxis=dict(range=[0, 100]),
            template="plotly_white"
        )
    else:
        chart = go.Figure()
        chart.update_layout(title="Prediction probability is unavailable")

    messages = {
        "Low": "Continue monitoring food preparation and actual waste.",
        "Medium": "Review the planned quantity and recent attendance.",
        "High": "Review the meal quantity and adjust using reliable records."
    }

    result = f"""
### Predicted Food Waste Risk: {prediction}

**Day:** {day}
**Menu:** {menu_type}
**Expected attendance:** {int(attendance)}
**Meals planned:** {int(meals_prepared)}
**Weather:** {weather}

**Suggested action:** {messages.get(prediction, 'Review the result carefully.')}

*Demonstration model: validate predictions with actual canteen waste records.*
"""

    return result, chart


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🍽️ Food Waste Risk Prediction Dashboard
        ### AI-based canteen meal planning
        Enter meal details to predict the expected food-waste risk.
        """
    )

    with gr.Row():
        with gr.Column():
            day = gr.Dropdown(
                ["Monday", "Tuesday", "Wednesday",
                 "Thursday", "Friday", "Saturday"],
                value="Monday",
                label="Day"
            )
            menu_type = gr.Dropdown(
                ["Veg", "Non-veg", "Snacks"],
                value="Veg",
                label="Menu Type"
            )

            attendance = gr.Number(
                value=450,
                minimum=1,
                precision=0,
                label="Expected Attendance",
                info="Enter the expected number of people"
            )


            weather = gr.Dropdown(
                ["Sunny", "Rainy", "Cloudy"],
                value="Rainy",
                label="Weather"
            )
            meals_prepared = gr.Number(
                 value=500,
                 minimum=1,
                 precision=0,
                 label="Meals to Prepare",
                 info="Enter the total number of meals to prepare"
            )

            predict_button = gr.Button("Predict Risk", variant="primary")

        with gr.Column():
            result = gr.Markdown(label="Prediction Result")
            chart_output = gr.Plot(label="Risk Probability Chart")

    predict_button.click(
        fn=predict_waste,
        inputs=[day, menu_type, attendance, weather, meals_prepared],
        outputs=[result, chart_output]
    )

demo.launch(share=True)


/tmp/ipykernel_1109/239604729.py:65: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://159e209a0879716179.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
